# T2.3 — Unified vs. Segment-Specific Statistical Comparison

**Research Task**: Compare a single pooled (unified) model that trains on all tea-market segments against individual segment-specific models.
Uses `TimeSeriesSplit` (5-fold), paired t-tests, Diebold-Mariano test, and produces a compact summary table ready for the paper.

## 1. Configuration & Imports

In [4]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)
pd.set_option('display.max_rows', 120)
np.random.seed(42)

# ── USER-CONFIGURABLE VARIABLES ─────────────────────────────────
DATA_PATH = None
TARGET_COL = 'price_mid_lkr'
SEGMENT_COL = 'category_type'
SALE_NO_COL = 'sale_number'
SALE_ID_COL = 'sale_id'
N_CV_FOLDS = 5
RANDOM_STATE = 42

def make_model():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('regressor', RandomForestRegressor(
            n_estimators=400,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

OUTPUT_DIR = None

print('Dependencies loaded.')

Dependencies loaded.


## 2. Load Data

In [5]:
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / '.git').exists():
            return p
        p = p.parent
    return Path.cwd()

repo_root = find_repo_root()
(repo_root / 'reports' / 'tables').mkdir(parents=True, exist_ok=True)
processed_dir = repo_root / 'data' / 'Processed'


_candidates = [
    processed_dir / 'tea_preprocessed_t13_supply_demand.csv',
    processed_dir / 'tea_preprocessed_v2.csv',
    processed_dir / 'tea_preprocessed.csv',
]
if DATA_PATH is None:
    DATA_PATH = next((p for p in _candidates if p.exists()), None)
    if DATA_PATH is None:
        raise FileNotFoundError('No preprocessed dataset found.')

if OUTPUT_DIR is None:
    OUTPUT_DIR = repo_root / 'reports' / 'tables'

df_raw = pd.read_csv(DATA_PATH)
print(f'Loaded: {DATA_PATH.name}  |  shape = {df_raw.shape}')

Loaded: tea_preprocessed_t13_supply_demand.csv  |  shape = (3034, 189)


## 3. Data Preparation & Sorting
Ensure that the data is chronologically ordered for accurate CV splitting and data leakage prevention.

In [6]:
df = df_raw.loc[df_raw[TARGET_COL].notna()].copy()
segments = sorted(df[SEGMENT_COL].dropna().unique())

if 'sale_year' in df.columns:
    df['_sale_year_num'] = pd.to_numeric(df['sale_year'], errors='coerce').fillna(0)
else:
    df['_sale_year_num'] = 0

df['_sale_num'] = pd.to_numeric(df[SALE_NO_COL], errors='coerce').fillna(0)
df = df.sort_values(['_sale_year_num', '_sale_num'], kind='mergesort').reset_index(drop=True)
df['_time_order'] = np.arange(len(df))

EXCLUDE_COLS = {
    SALE_ID_COL, 'sale_date_raw', 'sale_month', 'grade', 'tier',
    SEGMENT_COL, 'table_source', 'elevation', 'category',
    'price_lo_lkr', 'price_hi_lkr', 'price_range_lkr',
    TARGET_COL, 'price_mid_lkr_log', 'has_price_target', 'price_mid_usd',
    '_sale_year_num', '_sale_num', '_time_order',
    '_sale_date_parsed', '_sale_number_numeric', '_sale_order',
    '_current_supply_proxy', '_current_demand_proxy',
    '_historical_supply_avg', '_historical_demand_avg',
}

feature_cols = [
    c for c in df.columns
    if c not in EXCLUDE_COLS and pd.api.types.is_numeric_dtype(df[c])
]

X_all = df[feature_cols].values
y_all = df[TARGET_COL].values
segments_arr = df[SEGMENT_COL].values

print(f'Feature columns selected: {len(feature_cols)}')
print(f'Segments: {segments}')

Feature columns selected: 169
Segments: ['dust', 'high_grown', 'low_grown', 'off_grade']


## 4. Evaluation and Statistical Helper Functions

In [7]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    denom = np.where(np.abs(y_true) < 1e-9, np.nan, y_true)
    mape = float(np.nanmean(np.abs((y_true - y_pred) / denom)) * 100)
    return {'rmse': rmse, 'mae': mae, 'mape': mape, 'r2': r2}

def diebold_mariano_test(e1: np.ndarray, e2: np.ndarray, horizon: int = 1, power: int = 2) -> tuple[float, float]:
    d = np.abs(e1) ** power - np.abs(e2) ** power
    n = len(d)
    if n < 3:
        return np.nan, np.nan

    d_mean = np.mean(d)
    gamma_0 = np.var(d, ddof=1)
    gamma_sum = 0.0
    for k in range(1, horizon):
        gamma_k = np.cov(d[k:], d[:-k], ddof=1)[0, 1] if n > k else 0.0
        gamma_sum += gamma_k
    var_d = (gamma_0 + 2 * gamma_sum) / n

    if var_d <= 0:
        return np.nan, np.nan

    dm_stat = d_mean / np.sqrt(var_d)
    k = ((n + 1 - 2 * horizon + horizon * (horizon - 1) / n) / n) ** 0.5
    dm_stat_corrected = dm_stat * k if k > 0 else dm_stat
    p_value = 2.0 * stats.t.sf(np.abs(dm_stat_corrected), df=n - 1)
    return float(dm_stat_corrected), float(p_value)

## 5. Main Cross-Validation Pipeline
Compare predicting off a completely unified global model vs isolated individual segment models.

In [8]:
tscv = TimeSeriesSplit(n_splits=N_CV_FOLDS)
fold_metrics_rows = []
prediction_rows = []

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_all)):
    X_train_fold, X_test_fold = X_all[train_idx], X_all[test_idx]
    y_train_fold, y_test_fold = y_all[train_idx], y_all[test_idx]
    seg_train_fold = segments_arr[train_idx]
    seg_test_fold = segments_arr[test_idx]

    unified_model = make_model()
    unified_model.fit(X_train_fold, y_train_fold)
    unified_pred = unified_model.predict(X_test_fold)

    segment_pred = np.full_like(unified_pred, np.nan)
    for seg in segments:
        seg_train_mask = seg_train_fold == seg
        seg_test_mask = seg_test_fold == seg

        n_train_seg = seg_train_mask.sum()
        n_test_seg = seg_test_mask.sum()

        if n_test_seg == 0:
            continue

        if n_train_seg < 5:
            segment_pred[seg_test_mask] = unified_pred[seg_test_mask]
            continue

        seg_model = make_model()
        seg_model.fit(X_train_fold[seg_train_mask], y_train_fold[seg_train_mask])
        segment_pred[seg_test_mask] = seg_model.predict(X_test_fold[seg_test_mask])

    for i, global_idx in enumerate(test_idx):
        actual = y_test_fold[i]
        u_pred = unified_pred[i]
        s_pred = segment_pred[i]
        seg = seg_test_fold[i]
        prediction_rows.append({
            'fold': fold_idx + 1,
            'global_idx': int(global_idx),
            'segment': seg,
            'actual': actual,
            'pred_unified': u_pred,
            'pred_segment': s_pred,
            'error_unified': actual - u_pred,
            'error_segment': actual - s_pred if not np.isnan(s_pred) else np.nan,
        })

    for seg in segments:
        seg_mask = seg_test_fold == seg
        if seg_mask.sum() == 0:
            continue

        y_seg = y_test_fold[seg_mask]
        u_seg = unified_pred[seg_mask]
        s_seg = segment_pred[seg_mask]

        if np.all(np.isnan(s_seg)):
            continue

        m_uni = compute_metrics(y_seg, u_seg)
        m_seg = compute_metrics(y_seg, s_seg)

        fold_metrics_rows.append({
            'fold': fold_idx + 1,
            'segment': seg,
            'n_test': int(seg_mask.sum()),
            'rmse_unified': m_uni['rmse'],
            'mae_unified': m_uni['mae'],
            'mape_unified': m_uni['mape'],
            'r2_unified': m_uni['r2'],
            'rmse_segment': m_seg['rmse'],
            'mae_segment': m_seg['mae'],
            'mape_segment': m_seg['mape'],
            'r2_segment': m_seg['r2'],
        })

print('Cross-validation successfully executed.')

Cross-validation successfully executed.


## 6. Aggregate Fold Results and Test Statistics

In [9]:
df_fold_metrics = pd.DataFrame(fold_metrics_rows)
df_predictions = pd.DataFrame(prediction_rows)

summary_rows = []
for seg in segments:
    seg_folds = df_fold_metrics[df_fold_metrics['segment'] == seg]
    seg_preds = df_predictions[df_predictions['segment'] == seg]

    if len(seg_folds) < 2:
        continue

    rmse_u = seg_folds['rmse_unified'].values
    rmse_s = seg_folds['rmse_segment'].values
    mae_u = seg_folds['mae_unified'].values
    mae_s = seg_folds['mae_segment'].values
    r2_u = seg_folds['r2_unified'].values
    r2_s = seg_folds['r2_segment'].values

    rmse_u_mean = rmse_u.mean()
    rmse_s_mean = rmse_s.mean()

    pct_imp_rmse = ((rmse_u_mean - rmse_s_mean) / rmse_u_mean * 100 if rmse_u_mean > 0 else np.nan)

    t_stat, t_pval = stats.ttest_rel(rmse_u, rmse_s) if len(rmse_u) >= 2 else (np.nan, np.nan)

    e_uni = seg_preds['error_unified'].dropna().values
    e_seg = seg_preds['error_segment'].dropna().values
    min_len = min(len(e_uni), len(e_seg))
    dm_stat, dm_pval = diebold_mariano_test(e_uni[:min_len], e_seg[:min_len]) if min_len >= 5 else (np.nan, np.nan)

    if rmse_s_mean < rmse_u_mean:
        better = 'segment-specific'
    elif rmse_s_mean > rmse_u_mean:
        better = 'unified'
    else:
        better = 'tie'

    summary_rows.append({
        'segment': seg,
        'n_folds': len(seg_folds),
        'n_obs': len(seg_preds),
        'rmse_unified_mean': round(rmse_u_mean, 2),
        'rmse_segment_mean': round(rmse_s_mean, 2),
        'pct_improvement_rmse': round(pct_imp_rmse, 2),
        'r2_unified_mean': round(r2_u.mean(), 4),
        'r2_segment_mean': round(r2_s.mean(), 4),
        'paired_t_pvalue': round(t_pval, 6),
        'dm_pvalue': round(dm_pval, 6),
        'better_model': better,
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

,segment,n_folds,n_obs,rmse_unified_mean,rmse_segment_mean,pct_improvement_rmse,r2_unified_mean,r2_segment_mean,paired_t_pvalue,dm_pvalue,better_model
0,dust,5,441,199.33,199.56,-0.12,0.0744,0.0716,0.719225,0.757242,unified
1,high_grown,5,420,271.46,271.43,0.01,-0.0416,-0.0419,0.965293,0.958475,segment-specific
2,low_grown,5,1118,613.92,613.79,0.02,0.3310,0.3313,0.728743,0.860780,segment-specific
3,off_grade,5,426,172.89,172.63,0.15,-0.0187,-0.0157,0.424561,0.401168,segment-specific


## 7. Extract Table to Files

In [10]:
path_fold = OUTPUT_DIR / 'unified_vs_segment_fold_metrics.csv'
path_pred = OUTPUT_DIR / 'unified_vs_segment_predictions.csv'
path_summ = OUTPUT_DIR / 'unified_vs_segment_summary.csv'

df_fold_metrics.to_csv(path_fold, index=False)
df_predictions.to_csv(path_pred, index=False)
df_summary.to_csv(path_summ, index=False)

print("LaTeX Formatted Output:")
latex_cols = ['segment', 'rmse_unified_mean', 'rmse_segment_mean', 'pct_improvement_rmse', 'paired_t_pvalue', 'dm_pvalue', 'better_model']
print(df_summary[latex_cols].to_latex(index=False, escape=False, na_rep="-"))

LaTeX Formatted Output:
\begin{tabular}{lrrrrrl}
\toprule
segment & rmse_unified_mean & rmse_segment_mean & pct_improvement_rmse & paired_t_pvalue & dm_pvalue & better_model \\
\midrule
dust & 199.330000 & 199.560000 & -0.120000 & 0.719225 & 0.757242 & unified \\
high_grown & 271.460000 & 271.430000 & 0.010000 & 0.965293 & 0.958475 & segment-specific \\
low_grown & 613.920000 & 613.790000 & 0.020000 & 0.728743 & 0.860780 & segment-specific \\
off_grade & 172.890000 & 172.630000 & 0.150000 & 0.424561 & 0.401168 & segment-specific \\
\bottomrule
\end{tabular}

